#  CC2642R1TWFRTCRQ1 Firmware 
## Non-inertial Acceleration removal, Validity Checks & CSV writing/export prep



**Inputs:** JSON file produced by dsPIC33AK256MC205-H/M7 to replicate data arrays.
**Outputs:** Two CSV files - 100 Hz (vibration + RPM) and 1 Hz (inclination + temperature).

| Section | Purpose |
|---|---|
| 1 | Load JSON data from Notebook 2 |
| 2 | Inclination validity checks (re-derived from gyro stream) |
| 3 | Kinematic artefact removal (centripetal + Euler + Coriolis) |
| 4 | Rolling 1-second RMS for vibration channels |
| 5 | 100 Hz CSV parsing |
| 6 | 1 Hz CSV parsing |
| 7 | CSV file naming |
| 8 | CSV export |


---
## Section 0 - Imports


In [ ]:
import numpy as np
import pandas as pd
import json, os, datetime

# Validity threshold from the technical reference
OMEGA_VALID_THRESHOLD_DPS = 50.0

print("Imports loaded")


---
## Section 1 - Load JSON Data from Notebook 2

Reads the data-returns JSON written by the data-generation notebook. Returns a flat dict of numpy arrays keyed by the variable names from the upstream notebook.

The JSON arrays may be downsampled relative to the original sample rate. The meta block gives `duration_s`, `fs_high_hz` and `fs_low_hz` so a true elapsed-time vector can be reconstructed.


In [ ]:
def load_pipeline_data(filepath):
    """Load the data-returns JSON from Notebook 2.

    Args:
        filepath: path to the JSON file.

    Returns:
        dict with keys:
            meta              : run metadata (duration_s, rates, etc.)
            temp_cjc, temp_hjc          [1 Hz]
            omega_dps, rpm              [100 Hz]
            inclination, inclination_valid  [1 Hz]
            vib_x, vib_y, vib_z         [100 Hz]
            euler, coriolis             [100 Hz, (n, 3) arrays]
            a_centripetal               [100 Hz, (n, 3) array]
            t_high, t_low               reconstructed time vectors [s]
    """
    with open(filepath, "r") as f:
        j = json.load(f)

    meta = j["meta"]
    duration_s = meta["duration_s"]

    s4 = j["section_4_lm95172_temperature"]
    s5 = j["section_5_adxrs645_gyroscope"]
    s6 = j["section_6_adxl206_inclination"]
    s7 = j["section_7_at10tb_vibration"]
    s8 = j["section_8_kinematic_artefacts"]
    s9 = j["section_9_kinematic_correction"]

    out = {
        "meta"             : meta,
        "temp_cjc"         : np.asarray(s4["temp_cjc"]),
        "temp_hjc"         : np.asarray(s4["temp_hjc"]),
        "omega_dps"        : np.asarray(s5["omega_dps"]),
        "rpm"              : np.asarray(s5["rpm"]),
        "inclination"      : np.asarray(s6["inclination"]),
        "inclination_valid": np.asarray(s6["inclination_valid"]),
        "vib_x"            : np.asarray(s7["vib_x"]),
        "vib_y"            : np.asarray(s7["vib_y"]),
        "vib_z"            : np.asarray(s7["vib_z"]),
        "euler"            : np.asarray(s8["euler"]),
        "coriolis"         : np.asarray(s8["coriolis"]),
        "a_centripetal"    : np.asarray(s9["a_centripetal"]),
    }

    # Reconstruct time vectors from actual array lengths and duration
    n_high   = len(out["vib_x"])
    n_low    = len(out["temp_cjc"])
    out["t_high"] = np.linspace(0, duration_s, n_high)
    out["t_low"]  = np.linspace(0, duration_s, n_low)
    out["fs_high_effective"] = (n_high - 1) / duration_s
    out["fs_low_effective"]  = (n_low  - 1) / duration_s
    return out


# Point this at the JSON written by Notebook 2
JSON_INPUT_PATH = r'C:\Users\fm747\OneDrive - University of Bath\GBDP - Group 27 - Sensor IC Design\Complete Software Stack\Complete System Callibration and Firmware Design\ics_raw_gen_output\ICS_FW2_GEN_20260401_120000_data_returns.json'
data = load_pipeline_data(JSON_INPUT_PATH)

print(f"Loaded: {JSON_INPUT_PATH}")
print(f"  Run ID         : {data['meta']['run_id']}")
print(f"  Duration       : {data['meta']['duration_s']} s")
print(f"  100 Hz samples : {len(data['vib_x'])} (effective rate {data['fs_high_effective']:.2f} Hz)")
print(f"  1 Hz  samples  : {len(data['temp_cjc'])} (effective rate {data['fs_low_effective']:.2f} Hz)")


---
## Section 2 - Inclination Validity Check (Re-derived)

Per the technical reference, inclination is only valid when `|omega_z| < 50 dps` (tool near-stationary). Re-derive the flag from the gyro stream and cross-check against the value loaded from the JSON.


In [ ]:
def check_inclination_validity(omega_dps_high, fs_high, fs_low=1,
                                threshold_dps=OMEGA_VALID_THRESHOLD_DPS):
    """Re-derive the inclination_valid flag at 1 Hz from the high-rate gyro.

    A 1-second window is valid only if EVERY sample in that window has
    |omega| < threshold (conservative AND-reduction). Avoids surveys taken
    during ramp-up/ramp-down transients.

    Args:
        omega_dps_high : 100 Hz angular rate [deg/s], shape (n_high,)
        fs_high        : high-rate sample rate [Hz]
        fs_low         : low-rate sample rate [Hz] (default 1)
        threshold_dps  : maximum |omega| for validity [dps]

    Returns:
        valid : 1 = valid, 0 = suppressed, shape (n_low,)
    """
    samples_per_window = int(fs_high / fs_low)
    n_low              = len(omega_dps_high) // samples_per_window
    if n_low == 0:
        return np.zeros(0, dtype=int)

    trimmed = omega_dps_high[:n_low * samples_per_window]
    windows = trimmed.reshape(n_low, samples_per_window)
    valid   = (np.abs(windows) < threshold_dps).all(axis=1).astype(int)
    return valid


fs_high_eff = round(data["fs_high_effective"])
inclination_valid_rederived = check_inclination_validity(
    data["omega_dps"], fs_high=fs_high_eff
)

agreement = int((inclination_valid_rederived == data["inclination_valid"].astype(int)).sum())
total     = len(inclination_valid_rederived)

print(f"Re-derived inclination_valid : shape={inclination_valid_rederived.shape}")
print(f"  Valid windows        : {int(inclination_valid_rederived.sum())}/{total}")
print(f"  Agreement with JSON  : {agreement}/{total}")
print(f"  Threshold            : |omega| < {OMEGA_VALID_THRESHOLD_DPS} dps")


---
## Section 3 - Kinematic Artefact Removal

Subtract centripetal, Euler and Coriolis artefacts from the raw vibration channels. Per the technical reference, IEPE physics already removes the centripetal DC term from `vib_*`, so for the IEPE channels only Euler and Coriolis need explicit subtraction. Centripetal is included in the function signature so the same routine can also process the DC-coupled ADXL206HDZ inclination accelerometer.


In [ ]:
def remove_kinematic_artefacts(vib_x, vib_y, vib_z, a_centripetal, euler,
                                coriolis, include_centripetal=False):
    """Subtract kinematic artefacts from raw vibration channels.

    Args:
        vib_x, vib_y, vib_z : raw vibration [g], shape (n,)
        a_centripetal       : centripetal artefact [g], shape (n, 3)
        euler               : Euler artefact [g], shape (n, 3)
        coriolis            : Coriolis artefact [g], shape (n, 3)
        include_centripetal : True for DC-coupled accel (ADXL206), False for
                              IEPE (AT/10/TB) where centripetal is already
                              removed by AC coupling.

    Returns:
        vib_x_corr, vib_y_corr, vib_z_corr : kinematic-corrected channels [g]
    """
    artefact = euler + coriolis
    if include_centripetal:
        artefact = artefact + a_centripetal

    vib_x_corr = vib_x - artefact[:, 0]
    vib_y_corr = vib_y - artefact[:, 1]
    vib_z_corr = vib_z - artefact[:, 2]
    return vib_x_corr, vib_y_corr, vib_z_corr


# Apply to the IEPE channels (centripetal NOT subtracted - AC coupled)
vib_x_corr, vib_y_corr, vib_z_corr = remove_kinematic_artefacts(
    data["vib_x"], data["vib_y"], data["vib_z"],
    data["a_centripetal"], data["euler"], data["coriolis"],
    include_centripetal=False
)

print(f"Kinematic correction applied (IEPE - centripetal NOT subtracted)")
print(f"  vib_x_corr : range=[{vib_x_corr.min():+.4f}, {vib_x_corr.max():+.4f}] g")
print(f"  vib_y_corr : range=[{vib_y_corr.min():+.4f}, {vib_y_corr.max():+.4f}] g")
print(f"  vib_z_corr : range=[{vib_z_corr.min():+.4f}, {vib_z_corr.max():+.4f}] g")


---
## Section 4 - Rolling 1-Second RMS

Per the 100 Hz CSV schema, each axis needs a 1-second rolling RMS column. Window size = `fs_high` samples.


In [ ]:
def rolling_rms(signal, window):
    """1-second rolling RMS using a simple sliding window.

    On embedded hardware this is a circular buffer + running sum-of-squares.
    Here uses pandas for clarity at the boundary.
    """
    s = pd.Series(signal)
    return np.sqrt(s.pow(2).rolling(window, min_periods=1).mean()).to_numpy()


window_samples = max(1, fs_high_eff)
vib_x_rms = rolling_rms(vib_x_corr, window_samples)
vib_y_rms = rolling_rms(vib_y_corr, window_samples)
vib_z_rms = rolling_rms(vib_z_corr, window_samples)

print(f"Rolling RMS (window = {window_samples} samples = 1 s)")
print(f"  vib_x_rms : mean={vib_x_rms.mean():.4f} g")
print(f"  vib_y_rms : mean={vib_y_rms.mean():.4f} g")
print(f"  vib_z_rms : mean={vib_z_rms.mean():.4f} g")


---
## Section 5 - 100 Hz CSV Parsing

Build the 12-column 100 Hz dataframe matching the technical reference (Table 14).


In [ ]:
def build_100hz_dataframe(t_high, vib_x_raw, vib_y_raw, vib_z_raw,
                            vib_x_corr, vib_y_corr, vib_z_corr,
                            vib_x_rms, vib_y_rms, vib_z_rms,
                            rpm, omega_dps):
    """Build the 100 Hz dataframe in the exact column order from the spec."""
    return pd.DataFrame({
        "elapsed_time_s": np.round(t_high, 2),
        "vib_x_raw_g"   : vib_x_raw,
        "vib_y_raw_g"   : vib_y_raw,
        "vib_z_raw_g"   : vib_z_raw,
        "vib_x_corr_g"  : vib_x_corr,
        "vib_y_corr_g"  : vib_y_corr,
        "vib_z_corr_g"  : vib_z_corr,
        "vib_x_rms_g"   : vib_x_rms,
        "vib_y_rms_g"   : vib_y_rms,
        "vib_z_rms_g"   : vib_z_rms,
        "rpm"           : rpm,
        "omega_z_dps"   : omega_dps,
    })


df_100hz = build_100hz_dataframe(
    data["t_high"],
    data["vib_x"], data["vib_y"], data["vib_z"],
    vib_x_corr,    vib_y_corr,    vib_z_corr,
    vib_x_rms,     vib_y_rms,     vib_z_rms,
    data["rpm"],   data["omega_dps"]
)

print(f"100 Hz dataframe : {len(df_100hz)} rows x {len(df_100hz.columns)} cols")
print(f"Columns: {list(df_100hz.columns)}")


---
## Section 6 - 1 Hz CSV Parsing

Build the 6-column 1 Hz dataframe matching the technical reference. Uses the **re-derived** `inclination_valid` from Section 2, not the raw value from the JSON. RPM downsampled by averaging each 1-second window.


In [ ]:
def build_1hz_dataframe(n_low, inclination, inclination_valid,
                         temp_internal_c, temp_external_c, rpm_high, fs_high):
    """Build the 1 Hz dataframe in the exact column order from the spec.

    Args:
        n_low              : number of 1 Hz samples
        inclination        : inclination per second [deg]
        inclination_valid  : validity flag (re-derived from gyro)
        temp_internal_c    : LM95172 PCB temperature [C]
        temp_external_c    : MAX31856 borehole temperature [C]
        rpm_high           : 100 Hz RPM stream (will be averaged per second)
        fs_high            : high-rate sample rate [Hz]
    """
    samples_per_sec = int(fs_high)
    rpm_trim        = rpm_high[:n_low * samples_per_sec]
    rpm_1hz         = rpm_trim.reshape(n_low, samples_per_sec).mean(axis=1)

    return pd.DataFrame({
        "elapsed_time_s"   : np.arange(n_low, dtype=int),
        "inclination_deg"  : inclination,
        "inclination_valid": inclination_valid.astype(int),
        "temp_internal_c"  : temp_internal_c,
        "temp_external_c"  : temp_external_c,
        "rpm_1hz"          : rpm_1hz,
    })


n_low = len(data["temp_cjc"])
df_1hz = build_1hz_dataframe(
    n_low,
    data["inclination"],
    inclination_valid_rederived,
    data["temp_cjc"],
    data["temp_hjc"],
    data["rpm"],
    fs_high_eff,
)

print(f"1 Hz dataframe : {len(df_1hz)} rows x {len(df_1hz.columns)} cols")
print(f"Columns: {list(df_1hz.columns)}")
print(f"Valid inclination samples: {int(df_1hz['inclination_valid'].sum())}/{len(df_1hz)}")


---
## Section 7 - CSV File Naming

Filenames follow the technical reference convention: `ICS_<run_id>_100Hz.csv` and `ICS_<run_id>_1Hz.csv`. The run ID is taken from the input JSON meta block; if it already starts with `ICS_` the prefix is stripped to avoid duplication.


In [ ]:
def make_csv_filenames(run_id, output_dir):
    """Build the two CSV filenames per the technical reference convention.

    Args:
        run_id    : run identifier from the input JSON meta
        output_dir: directory to write into

    Returns:
        path_100hz, path_1hz : full output paths
    """
    csv_id = run_id[4:] if run_id.startswith("ICS_") else run_id
    path_100hz = os.path.join(output_dir, f"ICS_{csv_id}_100Hz.csv")
    path_1hz   = os.path.join(output_dir, f"ICS_{csv_id}_1Hz.csv")
    return path_100hz, path_1hz


OUTPUT_DIR = "./ics_fw3_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CSV_100HZ_PATH, CSV_1HZ_PATH = make_csv_filenames(data["meta"]["run_id"], OUTPUT_DIR)

print(f"100 Hz CSV path : {CSV_100HZ_PATH}")
print(f"1 Hz  CSV path  : {CSV_1HZ_PATH}")


---
## Section 8 - CSV Export

Write both files. Float columns formatted to 6 decimal places.


In [ ]:
def export_csv(df, filepath, float_format="%.6f"):
    """Write a dataframe to CSV with sane defaults."""
    df.to_csv(filepath, index=False, float_format=float_format)
    sz_kb = os.path.getsize(filepath) / 1024
    print(f"Wrote: {filepath}  ({sz_kb:.1f} kB, {len(df)} rows x {len(df.columns)} cols)")
    return filepath


export_csv(df_100hz, CSV_100HZ_PATH)
export_csv(df_1hz,   CSV_1HZ_PATH)

print("\n CSV export complete")
